# GEANT h96 reproducibility matrix3 — Drive-backed checkpoints

This notebook prepares and preflights a fresh, non-resumable matrix3. It does not train until the final cell is explicitly enabled and confirmed. Completed run directories are copied to Google Drive and read-back SHA-256 verified before matrix registration or the next run. Matrix lifecycle records are persisted as immutable snapshots after each registration.

The matrix excludes completed topology seed 42 and contains the same eight seed/mode pairs as matrix2. Scientific configuration, split boundaries, source/data identities, and v3 resource policy are unchanged. Test-split artifacts are excluded. Drive FUSE read-back verification is not a guarantee of cloud replication durability; retain the Drive copy and inspect it after the session. Do not resume partial matrix3 state.

In [ ]:
# Runtime pins are fixed; this cell does not authorize training.
import sys, subprocess, importlib.metadata as ilm
if sys.version_info[:2] != (3, 12): raise RuntimeError(f'Requires Python 3.12; found {sys.version.split()[0]}')
PINS = {'numpy':'1.26.4','torch':'2.12.0','psutil':'7.2.2','pytest':'9.1.1'}
subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check',*[f'{k}=={v}' for k,v in PINS.items()]],check=True)
actual = {k:ilm.version(k) for k in PINS}
if actual != PINS: raise RuntimeError(f'Pinned packages are not active: {actual}; restart runtime, then rerun from cell 1')
import torch
if torch.cuda.is_available(): raise RuntimeError('Select CPU runtime; CUDA is outside this protocol')
print('PASS: Python 3.12, exact package pins, CPU runtime', actual)

In [ ]:
# Upload and identify exactly four approved files by SHA-256. Filenames may have Colab (1) suffixes.
import hashlib, importlib, json, os, sys, uuid
from pathlib import Path
from google.colab import files, drive
EXPECTED = {
 'geant_h96_colab_source_v1.zip':'f4a35e77968d80069a04d027a793d118f6ecaa8ae9feb0a50e2fdd8c4736643d',
 'source_snapshot_manifest.json':'53789ea2deca2416781bd79c957e2598596ed7a55dd95555f932a62015d7bb15',
 'geant_h96_colab_trainval_v1.zip':'df2bac044c9b9a577d5117310a99022cce402b9370e2ba483ea18b8d7045521e',
 'dataset_bundle_manifest.json':'70f60f09b71c0cb89003ac2987af4433df0eeeb64b696845faa25d0fbadb8a95',
}
def digest(b): return hashlib.sha256(b).hexdigest()
uploaded = files.upload()
if len(uploaded) != 4: raise RuntimeError(f'Upload exactly four approved files; got {len(uploaded)}')
by_hash = {}
for _display_name, body in uploaded.items(): by_hash.setdefault(digest(body), []).append(body)
resolved = {}
for name, expected in EXPECTED.items():
    matches = by_hash.get(expected, [])
    if len(matches) != 1: raise RuntimeError(f'Missing, duplicate, or mismatched approved file: {name}')
    resolved[name] = matches[0]
if sum(len(v) for v in by_hash.values()) != 4: raise RuntimeError('Unexpected or duplicate upload refused')
SESSION_ID = uuid.uuid4().hex
SESSION = Path('/content') / f'geant_h96_matrix3_{SESSION_ID}'
SESSION.mkdir(exist_ok=False)
INCOMING = SESSION/'incoming'; INCOMING.mkdir(exist_ok=False)
for name, body in resolved.items():
    with (INCOMING/name).open('xb') as stream: stream.write(body); stream.flush(); os.fsync(stream.fileno())
source_zip = INCOMING/'geant_h96_colab_source_v1.zip'; source_manifest = INCOMING/'source_snapshot_manifest.json'
data_zip = INCOMING/'geant_h96_colab_trainval_v1.zip'; data_manifest = INCOMING/'dataset_bundle_manifest.json'
sys.path.insert(0, str(source_zip))
pkg = importlib.import_module('phase9_geant.colab_h96_v1.package')
source_check = pkg.verify_source_bundle(source_zip, source_manifest, EXPECTED[source_zip.name], EXPECTED[source_manifest.name])
dataset_check = pkg.verify_dataset_bundle(data_zip, data_manifest, EXPECTED[data_zip.name], EXPECTED[data_manifest.name])
source_obj = json.loads(source_manifest.read_text(encoding='utf-8'))
REPO = SESSION/'approved_source_snapshot'
pkg.extract_source_bundle(source_zip, REPO, source_obj)
sys.path.remove(str(source_zip)); importlib.invalidate_caches(); sys.path.insert(0, str(REPO))
pkg.verify_extracted_source(REPO, source_obj)
pkg.extract_dataset_bundle(data_zip, REPO)
DATA_ROOT = REPO/pkg.DATA_ROOT_RELATIVE
if {p.name for p in DATA_ROOT.iterdir()} != set(pkg.DATA_ARTIFACTS): raise RuntimeError('Dataset exact nine-file allowlist mismatch')
metadata = pkg._read_dataset_metadata(DATA_ROOT)
if metadata['windows']['train_count'] != 3924 or metadata['windows']['validation_count'] != 953: raise RuntimeError('h96 window counts mismatch')
from phase9_geant.colab_h96_v1 import environment, linux_resources, matrix_workflow
ENV = environment.verify_environment(require_pytest=True)
if linux_resources.RESOURCE_POLICY_ID != 'colab-linux-process-memory-v3-visible-cgroup' or linux_resources.PROCESS_PEAK_LIMIT_BYTES != 2*1024**3: raise RuntimeError('Expected v3 2-GiB memory policy is not active')
print('PASS: approved source/data digests, archive inventories, extraction, 9 train/validation members, and resource policy')
print({'source_members':len(source_check['members']),'dataset_members':len(dataset_check['members']),'train_windows':3924,'validation_windows':953,'test_members':0,'resource_policy':linux_resources.RESOURCE_POLICY_ID})

In [ ]:
# Mount Drive and choose a fresh, per-session persistent namespace. No run directory is created here.
drive.mount('/content/drive', force_remount=False)
DRIVE_BASE = Path('/content/drive/MyDrive/SDNResearch')
DRIVE_BASE.mkdir(parents=True, exist_ok=True)
DRIVE_ROOT = DRIVE_BASE/f'geant_h96_matrix3_{SESSION_ID}'
matrix_workflow.validate_persistent_matrix_target(DRIVE_ROOT)
print('PASS: Drive destination is available and fresh:', DRIVE_ROOT)
print('Known limitation: Google Drive FUSE read-back hashing checks the visible persistent copy, not remote replication durability.')

In [ ]:
# Complete the synthetic suite and a no-training preflight. Both use only approved source and synthetic fixtures.
import subprocess, uuid
TESTS = ['phase9_geant/colab_h96_v1/tests/test_portability.py','phase9_geant/colab_h96_v1/tests/test_launch_matrix.py','phase9_geant/colab_h96_v1/tests/test_matrix_workflow.py','phase9_geant/tests/test_horizon96_streaming_experiment.py','phase9_geant/tests/test_horizon96_streaming_experiment_v2.py','phase9_geant/tests/test_spatiotemporal_link.py']
test_base = SESSION/f'pytest_{uuid.uuid4().hex}'
test = subprocess.run([sys.executable,'-m','pytest','-q','-rs','--basetemp',str(test_base),*TESTS],cwd=REPO,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(test.stdout)
if test.returncode != 0: raise RuntimeError(f'Synthetic suite failed ({test.returncode}); no training is authorized')
ARGS = ['--repository-root',str(REPO),'--source-bundle',str(source_zip),'--source-manifest',str(source_manifest),'--expected-source-bundle-sha256',EXPECTED[source_zip.name],'--expected-source-manifest-sha256',EXPECTED[source_manifest.name],'--dataset-bundle',str(data_zip),'--dataset-manifest',str(data_manifest),'--expected-dataset-bundle-sha256',EXPECTED[data_zip.name],'--expected-dataset-manifest-sha256',EXPECTED[data_manifest.name],'--persistent-root',str(DRIVE_ROOT)]
LAUNCH = REPO/'phase9_geant/colab_h96_v1/launch_matrix.py'
dry = subprocess.run([sys.executable,'-I','-S',str(LAUNCH),'--dry-run',*ARGS],cwd=REPO,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(dry.stdout)
if dry.returncode != 0: raise RuntimeError(f'Matrix3 dry-run failed ({dry.returncode})')
DRY = json.loads(dry.stdout)
if DRY.get('status') != 'preflight_passed_no_training' or DRY.get('matrix_id') != 'geant-h96-reproducibility-matrix-v3-drive-backed': raise RuntimeError('Matrix3 dry-run identity/status mismatch')
if DRY.get('matrix_output_root') != 'geant_horizon96_streaming_experiment_v3_repro_matrix3' or DRY.get('persistent_root') != str(DRIVE_ROOT): raise RuntimeError('Matrix3 namespace mismatch')
if len(DRY.get('planned_runs', [])) != 8 or any(r['seed']==42 and r['adjacency_mode']=='topology' for r in DRY['planned_runs']): raise RuntimeError('Run plan does not exactly exclude the completed topology seed-42 run')
if DRY.get('resource_policy',{}).get('policy_id') != 'colab-linux-process-memory-v3-visible-cgroup': raise RuntimeError('Resource policy mismatch')
PREFLIGHT = {'status':DRY['status'],'matrix_id':DRY['matrix_id'],'persistent_root':str(DRIVE_ROOT),'planned_runs':DRY['planned_runs'],'source_sha256':EXPECTED[source_zip.name],'dataset_sha256':EXPECTED[data_zip.name],'resource_policy':DRY['resource_policy'],'test_split_accessed':False,'synthetic_suite_exit':test.returncode}
(SESSION/'preflight.json').write_text(json.dumps(PREFLIGHT,indent=2),encoding='utf-8')
print('PASS: matrix3 preflight only; eight-run plan confirmed; zero training runs launched')

## Explicit training authorization — disabled by default

Before enabling anything, independently review `data/transfer_bundles/geant_h96_colab_matrix_v1_review8/MATRIX3_SOURCE_AUDIT.json` and approve the exact source ZIP/manifest hashes. Confirm the displayed Drive path is the intended persistent location. To begin matrix3 only after that review, manually change both booleans below to `True` and type the exact phrase when prompted. Each run then requires its own exact seed/mode authorization. Any failure or partial Drive folder is preserved and blocks continuation; do not resume it. A new attempt requires a separately reviewed namespace/version.

In [ ]:
SOURCE_REVIEW_APPROVED = False
RUN_MATRIX3_AUTHORIZED = False
if SOURCE_REVIEW_APPROVED is not True or RUN_MATRIX3_AUTHORIZED is not True: raise PermissionError('Training remains disabled until separate source review and explicit authorization')
if PREFLIGHT.get('status') != 'preflight_passed_no_training' or PREFLIGHT.get('test_split_accessed') is not False: raise RuntimeError('No current successful matrix3 preflight receipt')
if input('Type AUTHORIZE GEANT H96 MATRIX3 to begin the eight-run matrix: ').strip() != 'AUTHORIZE GEANT H96 MATRIX3': raise PermissionError('No matrix authorization; nothing launched')
command = [sys.executable,'-I','-S',str(LAUNCH),'--run-matrix','--authorize-matrix3',*ARGS]
local_log = SESSION/'matrix3_launcher.log'
drive_log = DRIVE_BASE/f'geant_h96_matrix3_{SESSION_ID}.launcher.log'
with local_log.open('x',encoding='utf-8') as local, drive_log.open('x',encoding='utf-8') as persistent_log:
    def emit(line):
        print(line,flush=True)
        for stream in (local,persistent_log): stream.write(line+'\n'); stream.flush(); os.fsync(stream.fileno())
    def authorize_one(phrase): return input('Type exactly '+phrase+':\n')
    exit_code = matrix_workflow.run_matrix_with_prompt_bridge(command,prompt_fn=authorize_one,output_fn=emit)
if exit_code != 0: raise RuntimeError(f'Matrix3 stopped with exit {exit_code}; preserve local and Drive artifacts; do not resume')
print('Matrix3 completed. Review per-run backup_manifest.json files and immutable matrix_lifecycle snapshots in Drive:',DRIVE_ROOT)